In [ ]:
import requests
import tarfile
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import time

In [ ]:
def download_file(url, destination):
    """Download file with progress bar"""
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as f, tqdm(
            desc=destination.name,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for data in response.iter_content(chunk_size=1024):
                size = f.write(data)
                pbar.update(size)
        
        return True
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

def extract_archive(archive_path, extract_dir):
    """Extract tar.gz, tgz, or zip file"""
    try:
        if archive_path.suffix == '.gz' and archive_path.stem.endswith('.tar'):
            with tarfile.open(archive_path, 'r:gz') as tar:
                tar.extractall(extract_dir)
        elif archive_path.suffix == '.tgz':
            with tarfile.open(archive_path, 'r:gz') as tar:
                tar.extractall(extract_dir)
        elif archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
        else:
            print(f"Unsupported archive format: {archive_path.suffix}")
            return False
            
        print(f"Extracted: {archive_path.name}")
        return True
    except Exception as e:
        print(f"Error extracting {archive_path}: {e}")
        return False

def download_leipzig_all_languages(year='2019', corpus_size='100K'):
    """
    Download Leipzig Corpora for all 21 European languages
    
    Note: The Leipzig corpus URLs follow patterns but may change.
    You may need to manually browse and get exact URLs from:
    https://wortschatz.uni-leipzig.de/en/download/
    """
    
    # Language code mapping for Leipzig Corpora
    languages = {
        'Bulgarian': 'bul', 'Czech': 'ces', 'Danish': 'dan',
        'German': 'deu', 'Greek': 'ell', 'English': 'eng',
        'Estonian': 'est', 'Finnish': 'fin', 'French': 'fra',
        'Hungarian': 'hun', 'Italian': 'ita', 'Latvian': 'lav',
        'Lithuanian': 'lit', 'Dutch': 'nld', 'Polish': 'pol',
        'Portuguese': 'por', 'Romanian': 'ron', 'Slovak': 'slk',
        'Slovene': 'slv', 'Spanish': 'spa', 'Swedish': 'swe'
    }
    
    # Create directory structure
    base_dir = Path('data')
    leipzig_dir = base_dir / 'leipzig'
    leipzig_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 80)
    print("LEIPZIG CORPORA DOWNLOAD")
    print("=" * 80)
    
    base_urls = [
        "https://downloads.wortschatz-leipzig.de/corpora",
        "http://pcai056.informatik.uni-leipzig.de/downloads/corpora"
    ]
    
    successful_downloads = []
    failed_downloads = []
    
    for lang_name, lang_code in languages.items():
        print(f"\n{'='*60}")
        print(f"Processing: {lang_name} ({lang_code})")
        print(f"{'='*60}")
        
        # Try different filename patterns
        filename_patterns = [
            f"{lang_code}_news_{year}_{corpus_size}.tar.gz",
            f"{lang_code}-news-{year}-{corpus_size}.tar.gz",
            f"{lang_code}_newscrawl_{year}_{corpus_size}.tar.gz",
        ]
        
        downloaded = False
        
        for base_url in base_urls:
            if downloaded:
                break
                
            for filename in filename_patterns:
                url = f"{base_url}/{filename}"
                destination = leipzig_dir / filename
                
                # Skip if already downloaded
                if destination.exists():
                    print(f"✓ Already exists: {filename}")
                    successful_downloads.append(lang_name)
                    downloaded = True
                    break
                
                print(f"Trying: {url}")
                
                if download_file(url, destination):
                    # Extract the archive
                    extract_dir = leipzig_dir / lang_name
                    extract_dir.mkdir(exist_ok=True)
                    
                    if extract_archive(destination, extract_dir):
                        successful_downloads.append(lang_name)
                        downloaded = True
                        print(f"✓ Successfully downloaded and extracted {lang_name}")
                        break
                
                time.sleep(1)  # Be polite to the server
        
        if not downloaded:
            failed_downloads.append(lang_name)
            print(f"✗ Could not download {lang_name}")
    
    # Print summary
    print("\n" + "=" * 80)
    print("LEIPZIG DOWNLOAD SUMMARY")
    print("=" * 80)
    print(f"✓ Successfully downloaded: {len(successful_downloads)}/21")
    if successful_downloads:
        print(f"  Languages: {', '.join(successful_downloads)}")
    
    if failed_downloads:
        print(f"\n✗ Failed to download: {len(failed_downloads)}/21")
        print(f"  Languages: {', '.join(failed_downloads)}")
        print(f"\n  Please manually download these from:")
        print(f"  https://wortschatz.uni-leipzig.de/en/download/")
        print(f"  And place them in: {leipzig_dir.absolute()}")
    
    return leipzig_dir, successful_downloads, failed_downloads

def download_and_organize_europarl():
    """
    Download Europarl Parallel Corpus and organize by language
    """
    print("\n" + "=" * 80)
    print("EUROPARL CORPUS DOWNLOAD AND ORGANIZATION")
    print("=" * 80)
    
    base_dir = Path('data')
    europarl_dir = base_dir / 'europarl'
    europarl_dir.mkdir(parents=True, exist_ok=True)
    
    # Main Europarl v7 download URL
    europarl_url = "https://www.statmt.org/europarl/v7/europarl.tgz"
    destination = europarl_dir / "europarl.tgz"
    
    # Step 1: Download
    if destination.exists():
        print(f"✓ Europarl archive already exists: {destination}")
    else:
        print(f"Downloading Europarl corpus from: {europarl_url}")
        
        if download_file(europarl_url, destination):
            print("✓ Europarl download complete")
        else:
            print("✗ Europarl download failed")
            print("Please manually download from: http://www.statmt.org/europarl/")
            return europarl_dir, False
    
    # Step 2: Extract
    extracted_marker = europarl_dir / ".extracted"
    if not extracted_marker.exists():
        print("\nExtracting Europarl corpus...")
        if extract_archive(destination, europarl_dir):
            extracted_marker.touch()
            print("✓ Europarl extraction complete")
        else:
            print("✗ Europarl extraction failed")
            return europarl_dir, False
    else:
        print("✓ Europarl already extracted")
    
    # Step 3: Organize by language
    print("\n" + "=" * 80)
    print("ORGANIZING EUROPARL FILES BY LANGUAGE")
    print("=" * 80)
    
    txt_dir = europarl_dir / 'txt'
    txt_dir.mkdir(exist_ok=True)
    
    # All 21 language codes
    lang_codes = {
        'bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 
        'fi', 'fr', 'hu', 'it', 'lt', 'lv', 'nl', 'pl', 
        'pt', 'ro', 'sk', 'sl', 'sv'
    }
    
    # Create language directories
    for code in lang_codes:
        (txt_dir / code).mkdir(exist_ok=True)
    
    # Organize files
    organized_count = 0
    
    for lang_code in sorted(lang_codes):
        # Europarl v7 files are named: europarl-v7.XX-en.XX
        # where XX is the language code and the extension indicates content language
        
        # Try different possible patterns
        possible_patterns = [
            f"europarl-v7.{lang_code}-en.{lang_code}",  # e.g., europarl-v7.bg-en.bg
            f"europarl-v7.en-{lang_code}.{lang_code}",  # e.g., europarl-v7.en-bg.bg
        ]
        
        source_file = None
        
        for pattern in possible_patterns:
            potential_file = europarl_dir / pattern
            if potential_file.exists():
                source_file = potential_file
                break
        
        if source_file and source_file.exists():
            dest_file = txt_dir / lang_code / f"ep-{lang_code}.txt"
            
            if not dest_file.exists():
                shutil.copy2(source_file, dest_file)
                size_mb = source_file.stat().st_size / 1024 / 1024
                print(f"✓ {lang_code}: Organized ({size_mb:.2f} MB)")
                organized_count += 1
            else:
                print(f"⚠️  {lang_code}: Already organized")
                organized_count += 1
        else:
            print(f"✗ {lang_code}: Source file not found")
            print(f"    Tried: {', '.join(possible_patterns)}")
    
    print("\n" + "=" * 80)
    print(f"✓ Organized {organized_count}/{len(lang_codes)} languages")
    print("=" * 80)
    
    # Step 4: Verify organization
    print("\n" + "=" * 80)
    print("VERIFICATION: Checking language content")
    print("=" * 80)
    
    import re
    
    verification_samples = ['bg', 'cs', 'da', 'de', 'en', 'fr', 'es', 'sv']
    
    for lang_code in verification_samples:
        lang_file = txt_dir / lang_code / f"ep-{lang_code}.txt"
        
        if not lang_file.exists():
            print(f"✗ {lang_code}: File not found")
            continue
        
        # Read sample from middle of file
        with open(lang_file, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
            if len(lines) > 100:
                sample_line = lines[len(lines) // 2]
            else:
                sample_line = lines[0] if lines else ""
            
            # Clean sample
            sample_clean = re.sub(r'<[^>]+>', '', sample_line)
            sample_clean = ' '.join(sample_clean.split())[:80]
            
            print(f"{lang_code}: {sample_clean}...")
    
    return europarl_dir, True

def create_download_instructions():
    """
    Create a helper file with manual download instructions
    """
    instructions = """# Manual Download Instructions for Leipzig Corpora

If automatic download fails, please follow these steps:

1. Visit: https://wortschatz.uni-leipzig.de/en/download/

2. For each of the 21 languages, download the News corpus:
   - Bulgarian (bul)
   - Czech (ces)
   - Danish (dan)
   - German (deu)
   - Greek (ell)
   - English (eng)
   - Estonian (est)
   - Finnish (fin)
   - French (fra)
   - Hungarian (hun)
   - Italian (ita)
   - Latvian (lav)
   - Lithuanian (lit)
   - Dutch (nld)
   - Polish (pol)
   - Portuguese (por)
   - Romanian (ron)
   - Slovak (slk)
   - Slovene (slv)
   - Spanish (spa)
   - Swedish (swe)

3. Choose corpus type: "News Corpus"
4. Choose year: 2019 (or any available year - use 2020 for Slovak if 2019 unavailable)
5. Choose size: 100K, 300K, or 1M sentences

6. Save all downloaded .tar.gz files to: data/leipzig/

7. Run the extraction part of the script again.

For Europarl:
- Visit: http://www.statmt.org/europarl/
- Download the full corpus (europarl.tgz)
- Save to: data/europarl/
- Run the script again to extract and organize
"""
    
    with open('DOWNLOAD_INSTRUCTIONS.txt', 'w') as f:
        f.write(instructions)
    
    print("\n✓ Created DOWNLOAD_INSTRUCTIONS.txt")



In [5]:
# Define paths
data_dir = Path('data')
europarl_dir = data_dir / 'europarl'
europarl_archive = europarl_dir / 'europarl.tgz'

# Create directory if needed
europarl_dir.mkdir(parents=True, exist_ok=True)

# Check if archive exists
if europarl_archive.exists():
    size_mb = europarl_archive.stat().st_size / 1024 / 1024
    print(f"✓ europarl.tgz already exists")
    print(f"  Location: {europarl_archive.absolute()}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"✗ europarl.tgz not found at: {europarl_archive.absolute()}")
    print(f"  Please download from: http://www.statmt.org/europarl/v7/europarl.tgz")
    print(f"  And place it in: {europarl_dir.absolute()}")

✓ europarl.tgz already exists
  Location: c:\Users\addod\Applied AI Lab work\lab2_HD_SOM_BACKPROP\data\europarl\europarl.tgz
  Size: 1497.71 MB


In [ ]:
def download_europarl(url, destination):
    """Download europarl.tgz with progress bar"""
    try:
        print(f"Downloading from: {url}")
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as f, tqdm(
            desc="europarl.tgz",
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for data in response.iter_content(chunk_size=1024):
                size = f.write(data)
                pbar.update(size)
        
        print("✓ Download complete")
        return True
    except Exception as e:
        print(f"✗ Download failed: {e}")
        return False

# Download if not exists
if not europarl_archive.exists():
    europarl_url = "https://www.statmt.org/europarl/v7/europarl.tgz"
    download_europarl(europarl_url, europarl_archive)
else:
    print("✓ Archive already downloaded, skipping download")

In [12]:
# Check if already extracted
extracted_marker = europarl_dir / '.extracted'

if extracted_marker.exists():
    print("✓ Europarl already extracted")
else:
    print("Extracting europarl.tgz...")
    print("This may take a few minutes...")
    
    try:
        with tarfile.open(europarl_archive, 'r:gz') as tar:
            tar.extractall(europarl_dir)
        
        # Create marker file
        extracted_marker.touch()
        print("✓ Extraction complete")
    except Exception as e:
        print(f"✗ Extraction failed: {e}")

Extracting europarl.tgz...
This may take a few minutes...


C:\Users\addod\AppData\Local\Temp\ipykernel_28172\3506032452.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(europarl_dir)


✓ Extraction complete


In [13]:
import re

# Verify the txt directory structure
print("Verifying extraction...")
print("=" * 80)

txt_dir = europarl_dir / 'txt'

if not txt_dir.exists():
    print("txt/ folder not found!")
    print("Extraction may have failed or created different structure.")
    print("\nChecking what was extracted:")
    for item in europarl_dir.iterdir():
        if item.is_dir():
            print(f"  📁 {item.name}/")
        elif item.is_file() and not item.name.endswith('.tgz'):
            print(f"  📄 {item.name}")
else:
    print(f"✓ txt/ folder found at: {txt_dir.absolute()}")
    
    # Check language folders
    lang_folders = [f for f in txt_dir.iterdir() if f.is_dir()]
    
    if lang_folders:
        print(f"Found {len(lang_folders)} language folders")
        print(f"  Languages: {sorted([f.name for f in lang_folders])}")
        
        # Check files in each folder
        print("\nVerifying files in each language folder:")
        for lang_folder in sorted(lang_folders)[:5]:  # Show first 5
            files = list(lang_folder.glob('ep-*.txt'))
            print(f"  {lang_folder.name}: {len(files)} files")
            
            if files:
                # Read sample from first file
                with open(files[0], 'r', encoding='utf-8', errors='ignore') as f:
                    for line in f:
                        clean = re.sub(r'<[^>]+>', '', line).strip()
                        if clean and len(clean) > 20:
                            print(f"    Sample: {clean[:60]}...")
                            break
    else:
        print("No language folders found in txt/")

Verifying extraction...
✓ txt/ folder found at: c:\Users\addod\Applied AI Lab work\lab2_HD_SOM_BACKPROP\data\europarl\txt
Found 21 language folders
  Languages: ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hu', 'it', 'lt', 'lv', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

Verifying files in each language folder:
  bg: 6586 files
    Sample: Състав на Парламента: вж. протоколи...
  cs: 8842 files
    Sample: Schválení zápisu z předchozího zasedání: viz zápis...
  da: 9373 files
    Sample: Genoptagelse af sessionen...
  de: 9224 files
    Sample: Wiederaufnahme der Sitzungsperiode...
  el: 9271 files
    Sample: Επαvάληψη της συvσδoυ...


In [14]:
# Get detailed statistics
print("\nDetailed file count per language:")
print("=" * 80)

expected_langs = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 
                  'fi', 'fr', 'hu', 'it', 'lt', 'lv', 'nl', 'pl', 
                  'pt', 'ro', 'sk', 'sl', 'sv']

txt_dir = europarl_dir / 'txt'
found_langs = []
missing_langs = []
total_files = 0

for lang in expected_langs:
    lang_folder = txt_dir / lang
    
    if lang_folder.exists():
        files = list(lang_folder.glob('ep-*.txt'))
        file_count = len(files)
        total_files += file_count
        found_langs.append(lang)
        
        print(f"{lang}: {file_count:4d} files")
    else:
        missing_langs.append(lang)
        print(f"{lang}: Folder not found")

print("\n" + "=" * 80)
print(f"Summary:")
print(f"  Found languages: {len(found_langs)}/{len(expected_langs)}")
print(f"  Total ep-*.txt files: {total_files}")

if missing_langs:
    print(f"\nMissing languages: {', '.join(missing_langs)}")


Detailed file count per language:
bg: 6586 files
cs: 8842 files
da: 9373 files
de: 9224 files
el: 9271 files
en: 9672 files
es: 9433 files
et: 8819 files
fi: 9335 files
fr: 9450 files
hu: 8763 files
it: 9486 files
lt: 8819 files
lv: 8787 files
nl: 9433 files
pl: 8821 files
pt: 9434 files
ro: 6576 files
sk: 8804 files
sl: 8742 files
sv: 9402 files

Summary:
  Found languages: 21/21
  Total ep-*.txt files: 187072


In [15]:
print("\nFinal structure verification:")
print("=" * 80)

# Verify structure is ready for Lab2_main.ipynb
txt_dir = europarl_dir / 'txt'

if txt_dir.exists():
    lang_folders = [f for f in txt_dir.iterdir() if f.is_dir()]
    
    if lang_folders:
        # Count total files
        total_files = sum(len(list(f.glob('ep-*.txt'))) for f in lang_folders)
        
        print(f"✓ Structure is correct!")
        print(f"  Location: {txt_dir.absolute()}")
        print(f"  Language folders: {len(lang_folders)}")
        print(f"  Total ep-*.txt files: {total_files}")
        print(f"\n✓ Ready for preprocessing in Lab2_main.ipynb!")
    else:
        print("✗ No language folders found")
else:
    print("✗ txt/ folder not found")
    print("Please check extraction step")


Final structure verification:
✓ Structure is correct!
  Location: c:\Users\addod\Applied AI Lab work\lab2_HD_SOM_BACKPROP\data\europarl\txt
  Language folders: 21
  Total ep-*.txt files: 187072

✓ Ready for preprocessing in Lab2_main.ipynb!


In [ ]:
# Main execution
'''if __name__ == "__main__":
    print("=" * 80)
    print("STARTING DATASET DOWNLOAD FOR LAB 2")
    print("=" * 80)
    print("This will download and organize:")
    print("  1. Leipzig News Corpora (21 languages)")
    print("  2. Europarl Parallel Corpus (21 languages)")
    print("\nThis may take a while depending on your internet connection.\n")
    
    # Download Leipzig Corpora
    leipzig_dir, leipzig_success, leipzig_failed = download_leipzig_all_languages(
        year='2019',
        corpus_size='100K'
    )
    
    # Download and organize Europarl
    europarl_dir, europarl_success = download_and_organize_europarl()
    
    # Create manual download instructions
    # create_download_instructions()
    
    # Final summary
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    #print(f"Leipzig Corpora: {len(leipzig_success)}/21 languages downloaded")
    print(f"Europarl Corpus: {'✓ Success' if europarl_success else '✗ Failed'}")
    print(f"\nData directory: {Path('data').absolute()}")
    
    if leipzig_failed:
        print(f"\n{len(leipzig_failed)} languages need manual download:")
        print(f"   {', '.join(leipzig_failed)}")
        print("   See DOWNLOAD_INSTRUCTIONS.txt for details.")
    
    print("\n✓ Setup complete! You can now proceed with data preprocessing.")
    print("=" * 80)'''

STARTING DATASET DOWNLOAD FOR LAB 2
This will download and organize:
  1. Leipzig News Corpora (21 languages)
  2. Europarl Parallel Corpus (21 languages)

This may take a while depending on your internet connection.


EUROPARL CORPUS DOWNLOAD AND ORGANIZATION
✓ Europarl archive already exists: data\europarl\europarl.tgz

Extracting Europarl corpus...
Error extracting data\europarl\europarl.tgz: [WinError 183] Cannot create a file when that file already exists: 'data\\europarl\\txt\\fi'
✗ Europarl extraction failed

✓ Created DOWNLOAD_INSTRUCTIONS.txt

FINAL SUMMARY
Europarl Corpus: ✗ Failed

Data directory: c:\Users\addod\Applied AI Lab work\lab2_HD_SOM_BACKPROP\data

✓ Setup complete! You can now proceed with data preprocessing.


C:\Users\addod\AppData\Local\Temp\ipykernel_28172\3415823359.py:41: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)
